### **Prerequisites**
```
numpy==2.0.2
pandas==2.2.2
tokenizers==0.21.4
transformers==4.55.2
torch==2.8.0+cu126
```

- 만약, 기본 코랩과 버전이 다르다면 아래 명령어를 복사해서 실행시켜주세요.
```
%pip install numpy==2.0.2 pandas==2.2.2 tokenizers==0.21.4 transformers==4.55.2 torch==2.8.0+cu126 --index-url https://download.pytorch.org/whl
```

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from typing import (
    Generic,
    Tuple,
    TypeVar,
    List,
    Union,
    get_args
)
np.random.seed(1234)
torch.manual_seed(1234)
torch.cuda.manual_seed(1234)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

Batch = TypeVar("Batch", bound=int)
Token = TypeVar("Token", bound=int)
Sequence = TypeVar("Sequence", bound=int)
Layers = TypeVar("Layers", bound=int)
HiddenStates = TypeVar("HiddenStates", bound=int)
VocabSize = TypeVar("VocabSize", bound=int)
EmbeddingSize = TypeVar("EmbeddingSize", bound=int)
MaxLength = TypeVar("MaxLength", bound=int)

_1D = TypeVar("_1D")
_2D = TypeVar("_2D")
_3D = TypeVar("_3D")

def _label_str(self) -> str:
    oc = getattr(self, "__orig_class__", None)
    if oc is None:
        return "[]"
    args = get_args(oc)
    names = [getattr(a, "__name__", str(a)) for a in args]
    return "[" + ", ".join(names) + "]"


class Tensor1D(Generic[_1D]):
    def __init__(self, tensor: torch.Tensor):
        assert tensor.dim() == 1, ValueError("Tensor must be 1-dimensional")
        self.tensor = tensor
        self.s: _1D = tensor.size(0)  # sequence length

    def size(self) -> Tuple[int, int]:
        return self.tensor.size()

    def __repr__(self) -> str:
        return f"Tensor(shape=({self.s}))"

class Tensor2D(Generic[_1D, _2D]):
    def __init__(self, tensor: torch.Tensor):
        assert tensor.dim() == 2, ValueError("Tensor must be 2-dimensional")
        self.tensor = tensor
        self.b: _1D = tensor.size(0)  # batch size
        self.s: _2D = tensor.size(1)  # sequence length
        assert self.b == tensor.size(0), ValueError(
            f"Expected batch {self.b}, but got {tensor.size(0)}"
        )
        assert self.s == tensor.size(1), ValueError(
            f"Expected Sequence {self.s}, but got {tensor.size(1)}"
        )

    def size(self) -> Tuple[int, int]:
        return self.tensor.size()

    def __repr__(self) -> str:
        return f"Tensor(shape=({self.b}, {self.s}))"


class Tensor3D(Generic[_1D, _2D, _3D]):
    def __init__(self, tensor: torch.Tensor):
        assert tensor.dim() == 3, ValueError("Tensor must be 3-dimensional")
        self.tensor = tensor
        self.b: _1D = tensor.size(0)  # batch size
        self.s: _2D = tensor.size(1)  # sequence length
        self.h: _3D = tensor.size(2)  # hidden state size
        assert self.b == tensor.size(0), ValueError(
            f"Expected batch {self.b}, but got {tensor.size(0)}"
        )
        assert self.s == tensor.size(1), ValueError(
            f"Expected Sequence {self.s}, but got {tensor.size(1)}"
        )
        assert self.h == tensor.size(2), ValueError(
            f"Expected Hidden State {self.h}, but got {tensor.size(2)}"
        )

    def size(self) -> Tuple[int, int]:
        return self.tensor.size()

    def __repr__(self) -> str:
        return f"Tensor(shape=({self.b}, {self.s}, {self.h}))"


데이터셋을 확인

In [3]:
import pandas as pd
import os

file_list = os.listdir()
for file in file_list:
    if "ratings.txt" == file:
        print('학습에 필요한 파일이 존재합니다!', file)
        df = pd.read_table( (os.getcwd() + '/' + file), encoding='utf-8') # 데이터 프레임으로 보기 편하게 바꿔줍시다!
        df = df.dropna(how = 'any') # 널값을 없애줍니다!
        print('리뷰 갯수 :', len(df))
df.head()

학습에 필요한 파일이 존재합니다! ratings.txt
리뷰 갯수 : 199992


,id,document,label
0,8112052,어릴때보고 지금다시봐도 재밌어요ㅋㅋ,1
1,8132799,"디자인을 배우는 학생으로, 외국디자이너와 그들이 일군 전통을 통해 발전해가는 문화산...",1
2,4655635,폴리스스토리 시리즈는 1부터 뉴까지 버릴께 하나도 없음.. 최고.,1
3,9251303,와.. 연기가 진짜 개쩔구나.. 지루할거라고 생각했는데 몰입해서 봤다.. 그래 이런...,1
4,10067386,안개 자욱한 밤하늘에 떠 있는 초승달 같은 영화.,1


In [4]:
with open((os.getcwd() + '/' + 'naver_review.txt'), 'w', encoding='utf8') as f:
    # TODO: document 열만 가져와서 저장하는 코드를 구현합니다.
    f.write('\n'.join(df['document']))

In [ ]:
from tokenizers import BertWordPieceTokenizer

tokenizer = BertWordPieceTokenizer(
    lowercase=False,
    strip_accents=False,
)
tokenizer

Tokenizer(vocabulary_size=0, model=BertWordPiece, unk_token=[UNK], sep_token=[SEP], cls_token=[CLS], pad_token=[PAD], mask_token=[MASK], clean_text=True, handle_chinese_chars=True, strip_accents=False, lowercase=False, wordpieces_prefix=##)

In [6]:
data_file = 'naver_review.txt'
vocab_size = 30000
min_frequency = 2
initial_alphabet = []
limit_alphabet = 6000
special_tokens = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]
wordpieces_prefix = "##"
show_progress=True

tokenizer.train(
    files = data_file,
    vocab_size = vocab_size,
    min_frequency = min_frequency,
    initial_alphabet = initial_alphabet,
    limit_alphabet = limit_alphabet,
    special_tokens = special_tokens,
    wordpieces_prefix = wordpieces_prefix,
    show_progress = True,
)

vocab = tokenizer.get_vocab()
print("vocab size : ", len(vocab))
print(sorted(vocab, key=lambda x: vocab[x])[:20])




vocab size :  30000
['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/']


In [7]:
text = "I'm a student of SSAFY!"

encoded = tokenizer.encode(text)
print('🌱토큰화 결과 :',encoded.tokens)
print('🌱정수 인코딩 :',encoded.ids)
print('🌈디코딩 :',tokenizer.decode(encoded.ids))

🌱토큰화 결과 : ['I', "'", 'm', 'a', 'st', '##ud', '##ent', 'of', 'S', '##S', '##A', '##F', '##Y', '!']
🌱정수 인코딩 : [45, 11, 81, 69, 15444, 24882, 16071, 10280, 55, 3933, 4150, 4135, 3614, 5]
🌈디코딩 : I ' m a student of SSAFY!


In [8]:
embedding_vector = nn.Embedding()

TypeError: Embedding.__init__() missing 2 required positional arguments: 'num_embeddings' and 'embedding_dim'

In [9]:
embedding_vector: Tensor2D[VocabSize, EmbeddingSize] = nn.Embedding(vocab_size, 768)
embedding_vector.weight.shape

torch.Size([30000, 768])

그러면 특정 토큰의 임베딩 벡터를 확인해보겠습니다.

In [10]:
token_id = tokenizer.token_to_id("I")
print("token_id:", token_id)
input_id = torch.tensor([token_id], dtype=torch.long)
print("input_id 차원:", input_id.shape)

vector = embedding_vector(input_id)
print("vector 차원:", vector.shape)
print("vector:", vector)

token_id: 45
input_id 차원: torch.Size([1])
vector 차원: torch.Size([1, 768])
vector: tensor([[-5.8641e-01, -1.1327e+00,  2.6612e-02, -3.6936e-01, -4.5574e-01,
          1.4395e+00, -2.7049e-01, -2.3921e-02,  4.3165e-01,  6.3602e-01,
         -4.0117e-01, -1.0804e+00, -6.4650e-01, -6.8505e-02,  2.4397e-01,
         -2.0591e-01, -1.8770e-01,  4.2026e-01,  7.1682e-01, -5.9828e-01,
          3.1360e-01,  1.8200e+00,  2.8490e+00,  1.3980e+00,  1.0531e+00,
          2.0170e+00,  6.0673e-01, -1.5876e+00,  1.1668e+00, -3.1769e-01,
         -5.3360e-01, -4.7004e-01, -9.2409e-01,  1.3773e+00, -1.3743e-01,
          4.2839e-02, -4.8446e-01, -9.6651e-01, -1.5018e+00, -4.8411e-01,
          1.3622e+00, -1.7072e+00, -7.3317e-01,  2.9438e-01, -1.0314e+00,
          1.7281e+00,  1.4170e+00,  1.2014e-01, -1.5709e+00, -3.1901e-01,
         -2.0575e-02,  6.4082e-02, -1.9547e-01, -4.9615e-01,  4.1448e-01,
         -2.1306e-01, -8.5294e-02,  5.7862e-01,  9.8439e-02,  7.3975e-01,
          2.4581e-02,  9.2886e

In [11]:
word_embeddings: Tensor2D[VocabSize, EmbeddingSize] = nn.Embedding(vocab_size, 768)
print("워드 임베딩 차원 :", word_embeddings.weight.shape)

워드 임베딩 차원 : torch.Size([30000, 768])


In [ ]:
input_size: int = word_embeddings.weight.size()[1]
hidden_size: int = 1024 
num_layers: int = 1 
bidirectional: bool = False

rnn = nn.RNN(
    input_size=input_size,
    hidden_size=hidden_size,
    num_layers=num_layers,
    bidirectional=bidirectional
)

hidden_state_shape: int = (num_layers * (2 if bidirectional else 1), hidden_size)

h_0: Tensor2D[Sequence, HiddenStates] = torch.zeros(hidden_state_shape)
print("h_0의 차원 :",h_0.shape)

h_0의 차원 : torch.Size([1, 1024])


In [13]:
text: str = "나는 학교에 간다."

# 토큰화를 진행합니다.
encoded = tokenizer.encode(text)
# 토큰의 ids만 꺼냅니다.
input_ids: List[int] = encoded.ids

# 텐서화를 합니다.
input_ids: Tensor1D[Sequence] = torch.tensor(input_ids, dtype=torch.long)
input_ids

tensor([6227, 7125, 3279, 9046,   18])

In [14]:
input_embeds: Tensor2D[Sequence, EmbeddingSize] = word_embeddings(input_ids)
print("워드 임베딩 차원 : ", input_embeds.shape)  # (vocab_size, embedding_dim)
outputs = rnn(input_embeds, h_0)
hidden_states: Tensor2D[Sequence, HiddenStates] = outputs[0]
h_n: Tensor2D[Layers, HiddenStates] = outputs[1]

# sequence_length: input_token의 길이(length), hidden size: hidden state 차원 수, num_layers: layer 개수, num_dirs: 방향의 개수
print("hidden_states 차원 : ", hidden_states.shape)  # (sequence_length, d_h)
print("h_n 차원 : ", h_n.shape)  # (num_layers * num_dirs, d_h) = (1, d_h)

if torch.equal(hidden_states[-1].unsqueeze(0), h_n):
    print("hidden_states의 마지막과 h_n이 같습니다.")

워드 임베딩 차원 :  torch.Size([5, 768])
hidden_states 차원 :  torch.Size([5, 1024])
h_n 차원 :  torch.Size([1, 1024])
hidden_states의 마지막과 h_n이 같습니다.


In [ ]:
from abc import ABC, abstractmethod

class Encoder(nn.Module, ABC):
    def __init__(self: "Encoder") -> None:
        super().__init__()
        pass

    @abstractmethod
    def forward(self: "Encoder", input_ids: torch.Tensor) -> torch.Tensor:
        pass

class RNNEncoder(Encoder):
    def __init__(
        self: "RNNEncoder",
        vocab_size: int,
        embedding_dim: int,
        hidden_size: int,
        num_layers: int,
        bidirectional: bool,
    ) -> None:
        super().__init__()
        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            bidirectional=bidirectional,
        )
    
    def forward(
        self: "RNNEncoder",
        input_ids: Tensor1D[Sequence]
    ) -> Tuple[Tensor2D[Sequence, HiddenStates], Tensor2D[Layers, HiddenStates]]:
        input_embeds = self.word_embeddings(input_ids)
        outputs = self.rnn(input_embeds)
        hidden_states: Tensor2D[Sequence, HiddenStates] = outputs[0]
        h_n: Tensor2D[Layers, HiddenStates] = outputs[1]
        
        return hidden_states, h_n

vocab_size = 30000
embedding_dim = 768
hidden_size = 1024
num_layers = 1
bidirectional = False

rnn_encoder = RNNEncoder(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_size=hidden_size,
    num_layers=num_layers,
    bidirectional=bidirectional
)

outputs = rnn_encoder(input_ids)
hidden_states: Tensor2D[Sequence, HiddenStates] = outputs[0]
h_n: Tensor2D[Layers, HiddenStates] = outputs[1]
print("hidden_states 차원 : ", hidden_states.shape)
print("h_n 차원 : ", h_n.shape)


hidden_states 차원 :  torch.Size([5, 1024])
h_n 차원 :  torch.Size([1, 1024])


다음 디코더 부분을 구현하겠습니다.

In [ ]:
class Decoder(nn.Module, ABC):
    def __init__(self: "Decoder") -> None:
        super().__init__()

    @abstractmethod
    def forward(self, input_ids: torch.Tensor, init_hidden_state: torch.Tensor) -> torch.Tensor:
        pass

class RNNDecoder(Decoder):
    def __init__(
        self: "RNNDecoder",
        vocab_size: int,
        embedding_dim: int,
        hidden_size: int,
        num_layers: int,
        bidirectional: bool,
        start_token_id: int,
        end_token_id: int,
    ) -> None:
        super().__init__()
        self.start_token_id = start_token_id
        self.end_token_id = end_token_id
        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            bidirectional=bidirectional,
        )
        self.fully_connected_layer = nn.Linear(hidden_size, vocab_size)

    def forward(
        self: "RNNDecoder",
        init_hidden_state: Tensor2D[Layers, HiddenStates],
        max_len: int = 10
    ) -> Tuple[Tensor2D[MaxLength, VocabSize], List[int]]:
        logits: List[Tensor1D[VocabSize]] = []
        input_token: Tensor1D[Token] = torch.tensor([self.start_token_id], dtype=torch.long)
        output_token_ids: List[int] = [input_token.item()]
        h_n = init_hidden_state

        for _ in range(max_len):
            if input_token == self.end_token_id:
                break
            embedded: Tensor2D[Token, EmbeddingSize] = self.word_embeddings(input_token)
            outputs = self.rnn(embedded, h_n)
            h_n: Tensor2D[Layers, HiddenStates] = outputs[1]
            concat_h_n: Tensor1D[HiddenStates] = h_n.squeeze(0)

            """fully connected layer를 통해 [VocabSize]의 logit을 생성합니다."""
            logit: Tensor1D[VocabSize] = self.fully_connected_layer(concat_h_n)
            logits.append(logit)

            """logit 내에서 가장 높은 점수값을 가진 토큰을 선택합니다."""
            input_token: Tensor1D[Token] = torch.argmax(logit, dim=-1).unsqueeze(0)
            output_token_ids.append(input_token.item())
        
        """리스트의 logits를 torch의 Tensor로 변경합니다."""
        logits = torch.stack(logits, dim=0)

        return logits, output_token_ids

start_token_id: int = tokenizer.encode("[CLS]").ids[0]
end_token_id: int = tokenizer.encode("[SEP]").ids[0]

vocab_size: int = 30000
embedding_dim: int = 768
hidden_size: int = 1024
num_layers: int = 1
bidirectional: bool = False

rnn_decoder = RNNDecoder(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_size=hidden_size,
    num_layers=num_layers,
    bidirectional=bidirectional,
    start_token_id=start_token_id,
    end_token_id=end_token_id,
)
logits, output_token_ids = rnn_decoder(h_n)
output_texts = tokenizer.decode(output_token_ids)
print(output_texts)

합웃고읽스터모로 부탁합니다 슈퍼액션 짯 아파 공감되는


In [17]:
class RNNSeq2Seq(nn.Module):
    def __init__(self: "RNNSeq2Seq", encoder: nn.Module, decoder: nn.Module) -> None:
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self: "RNNSeq2Seq", input_ids: Tensor1D[Sequence]):
        hidden_states, context_vector = self.encoder(input_ids) # encoder에서 생성한 context_vector(h_n)을 decoder layer로 전달
        logits, output_tokens = self.decoder(context_vector)

        return logits, output_tokens

seq2seq = RNNSeq2Seq(rnn_encoder, rnn_decoder)
logits, output_tokens = seq2seq(input_ids)
output_token_ids = logits.argmax(dim=-1)
output_texts = tokenizer.decode(output_token_ids.tolist())
print(output_texts)

합웃고읽스터모로 부탁합니다 슈퍼액션 짯 아파 공감되는


In [ ]:
class LSTMEncoder(Encoder):
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int,
        hidden_size: int,
        num_layers: int,
        bidirectional: bool,
    ) -> None:
        super().__init__()
        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            bidirectional=bidirectional,
        )

    def forward(
        self: "LSTMEncoder",
        input_ids: Tensor1D[Sequence]
    )-> Tuple[
        Tensor2D[Sequence, HiddenStates],
        Tuple[
            Tensor2D[Layers, HiddenStates],
            Tensor2D[Layers, HiddenStates]
        ]
    ]:
        input_embeds = self.word_embeddings(input_ids)
        outputs = self.lstm(input_embeds)
        hidden_states: Tensor2D[Sequence, HiddenStates] = outputs[0]
        h_n: Tensor2D[Layers, HiddenStates] = outputs[1][0]
        c_n: Tensor2D[Layers, HiddenStates] = outputs[1][1]
        
        return hidden_states, (h_n, c_n)

vocab_size = 30000
embedding_dim = 768
hidden_size = 1024
num_layers = 1
bidirectional = False

lstm_encoder = LSTMEncoder(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_size=hidden_size,
    num_layers=num_layers,
    bidirectional=bidirectional
)

outputs = lstm_encoder(input_ids)
hidden_states: Tensor2D[Sequence, HiddenStates] = outputs[0]
h_n: Tensor2D[Layers, HiddenStates] = outputs[1][0]
c_n: Tensor2D[Layers, HiddenStates] = outputs[1][1]
print("hidden_states 차원 : ", hidden_states.shape)
print("h_n 차원 : ", h_n.shape)
print("c_n 차원 : ", c_n.shape) 



hidden_states 차원 :  torch.Size([5, 1024])
h_n 차원 :  torch.Size([1, 1024])
c_n 차원 :  torch.Size([1, 1024])


In [ ]:
class LSTMDecoder(Decoder):
    def __init__(
        self: "LSTMDecoder",
        vocab_size: int,
        embedding_dim: int,
        hidden_size: int,
        num_layers: int,
        bidirectional: bool,
        start_token_id: int,
        end_token_id: int,
    ) -> None:
        super().__init__()
        self.start_token_id = start_token_id
        self.end_token_id = end_token_id
        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            bidirectional=bidirectional,
        )
        self.fully_connected_layer = nn.Linear(hidden_size, vocab_size)

    def forward(
        self: "LSTMDecoder",
        init_hidden_state: Tensor2D[Layers, HiddenStates],
        init_cell_state: Tensor2D[Layers, HiddenStates],
        max_len: int = 10
    ) -> Tuple[Tensor2D[MaxLength, VocabSize], List[int]]:
        logits: List[Tensor1D[VocabSize]] = []
        input_token: Tensor1D[Token] = torch.tensor([self.start_token_id], dtype=torch.long)
        output_token_ids: List[int] = [input_token.item()]
        h_n = init_hidden_state
        c_n = init_cell_state

        for _ in range(max_len):
            if input_token == self.end_token_id:
                break

            embedded: Tensor2D[Token, EmbeddingSize] = self.word_embeddings(input_token)
            outputs = self.lstm(embedded, (h_n, c_n))
            h_n: Tensor2D[Layers, HiddenStates] = outputs[1][0]
            c_n: Tensor2D[Layers, HiddenStates] = outputs[1][1]

            concat_h_n: Tensor1D[HiddenStates] = h_n.squeeze(0)

            logit: Tensor1D[VocabSize] = self.fully_connected_layer(concat_h_n)
            logits.append(logit)

            input_token: Tensor1D[Token] = torch.argmax(logit, dim=-1).unsqueeze(0)
            output_token_ids.append(input_token.item())
        
        logits = torch.stack(logits, dim=0)  # [max_len, vocab_size]

        return logits, output_token_ids


start_token_id: int = tokenizer.encode("[CLS]").ids[0]
end_token_id: int = tokenizer.encode("[SEP]").ids[0]

vocab_size: int = 30000
embedding_dim: int = 768
hidden_size: int = 1024
num_layers: int = 1
bidirectional: bool = False

lstm_decoder = LSTMDecoder(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_size=hidden_size,
    num_layers=num_layers,
    bidirectional=bidirectional,
    start_token_id=start_token_id,
    end_token_id=end_token_id,
)

logits, output_tokens = lstm_decoder(h_n, c_n)
output_token_ids = logits.argmax(dim=-1)
output_texts = tokenizer.decode(output_token_ids.tolist())
print(output_texts)

아이고 추억의 지진랑은내리는 터널 터널 할말멩 잔


In [20]:
class LSTMSeq2Seq(nn.Module):
    def __init__(self: "LSTMSeq2Seq", encoder: nn.Module, decoder: nn.Module) -> None:
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self: "LSTMSeq2Seq", input_ids: Tensor1D[Sequence]):
        hidden_states, (context_vector, cell_states) = self.encoder(input_ids) # encoder에서 생성한 context_vector(h_n)을 decoder layer로 전달
        logits, output_tokens = self.decoder(context_vector, cell_states)

        return logits, output_tokens

seq2seq = LSTMSeq2Seq(lstm_encoder, lstm_decoder)
logits, output_tokens = seq2seq(input_ids)
output_token_ids = logits.argmax(dim=-1)
output_texts = tokenizer.decode(output_token_ids.tolist())
print(output_texts)

아이고 추억의 지진랑은내리는 터널 터널 할말멩 잔


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LuongAttention(nn.Module):
    def __init__(self: "LuongAttention", hidden_size: int):
        super().__init__()
        self.W_a = nn.Linear(hidden_size, hidden_size, bias=False)

    @torch.no_grad()  # 학습 시 제거하세요
    def forward(
        self:"LuongAttention",
        h_t: Tensor1D[HiddenStates],
        encoder_outputs: Tensor2D[Sequence, HiddenStates],
    ) -> Tuple[Tensor1D[HiddenStates], Tensor1D[Sequence]]:
        Wa_ht: Tensor1D[HiddenStates] = self.W_a(h_t)

        attention_score: Tensor1D[Sequence] = encoder_outputs @ Wa_ht
        attention_weights: Tensor1D[Sequence] = F.softmax(attention_score, dim=-1)
        context_vector: Tensor1D[HiddenStates] = attention_weights @ encoder_outputs

        return context_vector, attention_weights


In [ ]:
class AttentionDecoder(nn.Module):
    def __init__(
        self: "AttentionDecoder",
        vocab_size: int,
        embedding_dim: int,
        hidden_size: int,
        num_layers: int,
        bidirectional: bool,
        start_token_id: int,
        end_token_id: int,
    ):
        super().__init__()
        self.start_token_id = start_token_id
        self.end_token_id = end_token_id
        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            bidirectional=bidirectional,
        )
        
        self.attn = LuongAttention(hidden_size)
        self.W_c = nn.Linear(hidden_size * 2, hidden_size)
        self.fully_connected_layer = nn.Linear(hidden_size, vocab_size)

    @torch.no_grad()
    def forward(
        self:"AttentionDecoder",
        init_hidden_state: Tensor1D[HiddenStates],
        init_cell_state: Tensor1D[HiddenStates],
        encoder_outputs: Tensor2D[Sequence, HiddenStates],
        max_len: int = 10,
    ):
        logits: List[Tensor1D[VocabSize]] = []
        input_token: Tensor1D[Token] = torch.tensor([self.start_token_id], dtype=torch.long)
        output_token_ids: List[int] = [input_token.item()]
        h_n = init_hidden_state
        c_n = init_cell_state

        for _ in range(max_len):
            if input_token == self.end_token_id:
                break

            embedded: Tensor2D[Token, EmbeddingSize] = self.word_embeddings(input_token)
            outputs = self.lstm(embedded, (h_n, c_n))
            h_n: Tensor2D[Layers, HiddenStates] = outputs[1][0]
            c_n: Tensor2D[Layers, HiddenStates] = outputs[1][1]

            concat_h_n: Tensor1D[HiddenStates] = h_n.squeeze(0)

            # 어텐션
            context_vector, attention_weights = self.attn(concat_h_n, encoder_outputs)

            v_t: Tensor1D[HiddenStates * 2] = torch.cat([concat_h_n, context_vector], dim=-1)
            
            attentional_hidden_state: Tensor1D[HiddenStates] = torch.tanh(self.W_c(v_t))
            
            logit: Tensor1D[VocabSize] = self.fully_connected_layer(attentional_hidden_state)
            logits.append(logit)

            input_token: Tensor1D[Token] = torch.argmax(logit, dim=-1).unsqueeze(0)
            output_token_ids.append(input_token.item())

        logits = torch.stack(logits, dim=0) if logits else torch.empty(0, self.out.out_features)
        
        return logits, output_token_ids
    
start_token_id: int = tokenizer.encode("[CLS]").ids[0]
end_token_id: int = tokenizer.encode("[SEP]").ids[0]

vocab_size: int = 30000
embedding_dim: int = 768
hidden_size: int = 1024
num_layers: int = 1
bidirectional: bool = False

attention_decoder = AttentionDecoder(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_size=hidden_size,
    num_layers=num_layers,
    bidirectional=bidirectional,
    start_token_id=start_token_id,
    end_token_id=end_token_id,
)

logits, output_tokens = attention_decoder(h_n, c_n, hidden_states)
output_token_ids = logits.argmax(dim=-1)
output_texts = tokenizer.decode(output_token_ids.tolist())
print(output_texts)

##방울 인과 갈피를 시켰하겠네 난잡한퓨이었는데미네 갈리는


In [23]:
class AttentionSeq2Seq(nn.Module):
    def __init__(self: "AttentionSeq2Seq", encoder: nn.Module, decoder: nn.Module) -> None:
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self: "AttentionSeq2Seq", input_ids: Tensor1D[Sequence]):
        hidden_states, (last_hidden_state, cell_states) = self.encoder(input_ids) # encoder에서 생성한 h_n을 decoder layer로 전달
        logits, output_tokens = self.decoder(last_hidden_state, cell_states, hidden_states)

        return logits, output_tokens

seq2seq = AttentionSeq2Seq(lstm_encoder, attention_decoder)
logits, output_tokens = seq2seq(input_ids)
output_token_ids = logits.argmax(dim=-1)
output_texts = tokenizer.decode(output_token_ids.tolist())
print(output_texts)

##방울 인과 갈피를 시켰하겠네 난잡한퓨이었는데미네 갈리는


In [24]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "Helsinki-NLP/opus-mt-ko-en"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

/Users/seonjong/miniconda3/envs/c10/lib/python3.11/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [25]:
print(model)

MarianMTModel(
  (model): MarianModel(
    (shared): Embedding(65001, 512, padding_idx=65000)
    (encoder): MarianEncoder(
      (embed_tokens): Embedding(65001, 512, padding_idx=65000)
      (embed_positions): MarianSinusoidalPositionalEmbedding(512, 512)
      (layers): ModuleList(
        (0-5): 6 x MarianEncoderLayer(
          (self_attn): MarianAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation_fn): SiLU()
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=512, bias=True)
          (final_layer_norm): LayerNorm((512,), eps=1e-05

In [26]:
for name, param in model.named_parameters():
    print(name)

model.shared.weight
model.encoder.embed_positions.weight
model.encoder.layers.0.self_attn.k_proj.weight
model.encoder.layers.0.self_attn.k_proj.bias
model.encoder.layers.0.self_attn.v_proj.weight
model.encoder.layers.0.self_attn.v_proj.bias
model.encoder.layers.0.self_attn.q_proj.weight
model.encoder.layers.0.self_attn.q_proj.bias
model.encoder.layers.0.self_attn.out_proj.weight
model.encoder.layers.0.self_attn.out_proj.bias
model.encoder.layers.0.self_attn_layer_norm.weight
model.encoder.layers.0.self_attn_layer_norm.bias
model.encoder.layers.0.fc1.weight
model.encoder.layers.0.fc1.bias
model.encoder.layers.0.fc2.weight
model.encoder.layers.0.fc2.bias
model.encoder.layers.0.final_layer_norm.weight
model.encoder.layers.0.final_layer_norm.bias
model.encoder.layers.1.self_attn.k_proj.weight
model.encoder.layers.1.self_attn.k_proj.bias
model.encoder.layers.1.self_attn.v_proj.weight
model.encoder.layers.1.self_attn.v_proj.bias
model.encoder.layers.1.self_attn.q_proj.weight
model.encoder.la

In [27]:
text = "나는 학교에 간다."
"""여기서는 batch로 입력을 처리하여 차원이 [seq_len]이 아닌 [batch_size, seq_len]입니다. 여기서는 입력이 한개이므로 [1, seq_len]입니다."""
encoded = tokenizer(text, return_tensors="pt")

generated_ids = model.generate(
    **encoded,
    max_new_tokens=64,
)

translation = tokenizer.decode(generated_ids.squeeze(), skip_special_tokens=True)
print("SRC:", text)
print("MT :", translation)

SRC: 나는 학교에 간다.
MT : I'm going to school.


In [28]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

model_name = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)

Some weights of the model checkpoint at bert-base-cased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
sentence = f"I {tokenizer.mask_token} to school."

top_k = 5

encoded = tokenizer(sentence, return_tensors="pt", return_attention_mask=True)

input_ids = encoded.input_ids

mask_token_id = tokenizer.mask_token_id

mask_positions = (input_ids == mask_token_id).nonzero(as_tuple=False)

outputs = model(**encoded)

logits = outputs.logits.squeeze(0)

all_token_candidates: List[List[Tuple[str, float]]] = []
for _, pos in mask_positions:
    pos = pos.item()
    logits_at_pos = logits[pos]
    probs = torch.softmax(logits_at_pos, dim=-1)
    topk = torch.topk(probs, k=top_k)

    ids = topk.indices.tolist()
    scores = topk.values.tolist()
    tokens = [tokenizer.convert_ids_to_tokens(tid) for tid in ids]
    candidates = list(zip(tokens, scores))
    all_token_candidates.append(candidates)

restored_sentences: List[str] = []

token_candidates: List[Tuple[str, float]] = all_token_candidates[0]

for tok, _ in token_candidates:
    new_ids = input_ids.clone() 
    tok_id = tokenizer.convert_tokens_to_ids(tok)
    new_ids[0, mask_positions[0, 1]] = tok_id
    text = tokenizer.decode(new_ids[0], skip_special_tokens=True)
    restored_sentences.append(text.strip())

print("원본 문장:", sentence)
print("BERT가 예측한 문장들:")
for idx, sent in enumerate(restored_sentences, start=1):
    print("{}순위: {}".format(idx, sent))

원본 문장: I [MASK] to school.
BERT가 예측한 문장들:
1순위: I went to school.
2순위: I go to school.
3순위: I walked to school.
4순위: I ran to school.
5순위: I got to school.


In [30]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

In [31]:
prompt = "Once upon a time in a small village, a curious child found a mysterious key."
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=64,
    )

output_tokens = tokenizer.decode(generated_ids.squeeze(), skip_special_tokens=True)
print(output_tokens)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Once upon a time in a small village, a curious child found a mysterious key. The child was a boy named Kiyoshi. He was a boy who had been born with a strange, mysterious, and mysterious voice. He was a boy who had been born with a strange, mysterious, and mysterious voice. He was a boy who had been born with a strange, mysterious, and mysterious voice.
